# ResNet50 - UMAP - HBDSCAN for Cleaning Crude Data & Basic Clustering

#### Warning: ResNet50 is not very good at creating embeddings for BM images (ofc). Suggestion: Use Contrastive Learning for initial clustering

# Pipeline Code

In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
from torchvision import models, transforms
import umap.umap_ as umap
import hdbscan
import pandas as pd
import shutil

# Define paths
image_folder = "/Users/user/Desktop/NonFKwork/AI/DCIM"
output_csv_path = "/Users/user/Desktop/NonFKwork/AI/umap_clusters.csv"

# 1. Load pretrained ResNet50 model (without final layer)
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()

# 2. Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. Feature extraction function
def extract_features(img_path):
    try:
        img = Image.open(img_path).convert('RGB')
        input_tensor = preprocess(img).unsqueeze(0)
        with torch.no_grad():
            features = resnet(input_tensor)
        return features.squeeze().numpy()
    except Exception as e:
        print(f"Error reading {img_path}: {e}")
        return None

# 4. Process all images
image_paths = []
features = []

print("Extracting features...")
for fname in tqdm(os.listdir(image_folder)):
    if fname.lower().endswith(('.jpg', '.jpeg')):
        path = os.path.join(image_folder, fname)
        feat = extract_features(path)
        if feat is not None:
            features.append(feat)
            image_paths.append(path)

features = np.array(features)

# 5. Apply UMAP
print("Running UMAP...")
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine')
X_umap = umap_model.fit_transform(features)

# 6. Apply HDBSCAN clustering
print("Clustering with HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=15)
labels = clusterer.fit_predict(X_umap)

# 7. Save results
print("Saving output...")
df = pd.DataFrame({
    'image_path': image_paths,
    'umap_x': X_umap[:, 0],
    'umap_y': X_umap[:, 1],
    'cluster': labels
})
df.to_csv(output_csv_path, index=False)
print(f"Saved UMAP+cluster results to {output_csv_path}")


# Paths
csv_path = "/Users/user/Desktop/NonFKwork/AI/umap_clusters.csv"
output_base = "/Users/user/Desktop/NonFKwork/AI/clusters"

# Load CSV
df = pd.read_csv(csv_path)

# Create cluster folders and copy files
for cluster_id in df['cluster'].unique():
    cluster_folder = os.path.join(output_base, f"cluster_{cluster_id}")
    os.makedirs(cluster_folder, exist_ok=True)

    # Filter rows for this cluster
    cluster_df = df[df['cluster'] == cluster_id]

    for _, row in cluster_df.iterrows():
        src_path = row['image_path']
        if os.path.exists(src_path):
            filename = os.path.basename(src_path)
            dst_path = os.path.join(cluster_folder, filename)
            try:
                shutil.copy(src_path, dst_path)
            except Exception as e:
                print(f"Error copying {src_path}: {e}")
        else:
            print(f"File not found: {src_path}")

print(f"✅ Images copied to cluster folders in: {output_base}")

# For making Cluster Map

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
import os

# --- Configuration ---
csv_path = "/Users/user/Desktop/NonFKwork/AI/umap_clusters.csv"
sample_size = 200
image_zoom = 0.05
output_path = "/Users/user/Desktop/NonFKwork/AI/umap_thumbnail_plot2.png"

# --- Load and sample data ---
df = pd.read_csv(csv_path)
df_sampled = df.sample(n=min(sample_size, len(df)), random_state=42)

# --- Get axis limits from sample ---
x_min, x_max = df_sampled['umap_x'].min(), df_sampled['umap_x'].max()
y_min, y_max = df_sampled['umap_y'].min(), df_sampled['umap_y'].max()

# --- Helper to get image thumbnails ---
def get_thumbnail(image_path, zoom=0.2):
    try:
        img = Image.open(image_path).convert("RGB")
        return OffsetImage(img, zoom=zoom)
    except Exception as e:
        print(f"Error loading image: {image_path} - {e}")
        return None

# --- Set up the plot ---
fig, ax = plt.subplots(figsize=(20, 15))
ax.set_xlim(x_min - 1, x_max + 1)
ax.set_ylim(y_min - 1, y_max + 1)
ax.set_title("UMAP Projection with Sample Thumbnails", fontsize=18)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

# --- Add thumbnails to plot ---
for _, row in df_sampled.iterrows():
    x, y = row['umap_x'], row['umap_y']
    image_path = row['image_path']
    if os.path.exists(image_path):
        img_thumb = get_thumbnail(image_path, zoom=image_zoom)
        if img_thumb:
            ab = AnnotationBbox(img_thumb, (x, y), frameon=False)
            ax.add_artist(ab)
    else:
        print(f"⚠️ File not found: {image_path}")

# --- Save and show ---
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved UMAP thumbnail plot to: {output_path}")
